In [ ]:
# Customer Churn Prediction System - Part 3: Model Training...
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
import sys
import shap

sys.path.append(os.path.abspath('..'))

In [ ]:
# 1. Train-Test Split and Preprocessing
from sklearn.model_selection import train_test_split
from src.feature_engineering import ChurnFeatureEngineer

df = pd.read_csv("../data/processed/cleaned_churn.csv")

X = df.drop(columns=['Churn Value'])
y = df['Churn Value']

fe = ChurnFeatureEngineer(encoder_path="../models/encoder.joblib", scaler_path="../models/scaler.joblib")
X_engineered = fe.create_features(X)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_engineered, y, test_size=0.2, random_state=42, stratify=y
)

X_train = fe.fit_transform(X_train_raw)
X_test = fe.transform(X_test_raw)
fe.save_assets()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
# 2. Model Evaluation Function
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

def evaluate(model, X_train, y_train, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    metrics = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall (Churn)': recall_score(y_test, y_pred, zero_division=0),
        'F1 Score': f1_score(y_test, y_pred, zero_division=0),
        'ROC AUC': roc_auc_score(y_test, y_prob)
    }
    return metrics

In [ ]:
# 3. Train & Compare Multiple Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

comparison_list = []
for name, model in models.items():
    model.fit(X_train, y_train)
    metrics = evaluate(model, X_train, y_train, X_test, y_test, name)
    comparison_list.append(metrics)

df_comp = pd.DataFrame(comparison_list)
df_comp

In [ ]:
# 4. Hyperparameter Tuning using GridSearchCV
from sklearn.model_selection import GridSearchCV

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={
        'n_estimators': [100, 200],
        'max_depth': [5, 10, None],
        'class_weight': ['balanced', None]
    },
    scoring='f1',
    cv=3,
    n_jobs=-1
)
rf_grid.fit(X_train, y_train)
best_rf = rf_grid.best_estimator_
print("Best RF Params:", rf_grid.best_params_)

xgb_grid = GridSearchCV(
    XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    param_grid={
        'n_estimators': [100, 200],
        'max_depth': [3, 5, 7],
        'learning_rate': [0.01, 0.1, 0.2],
        'scale_pos_weight': [1, 3]
    },
    scoring='f1',
    cv=3,
    n_jobs=-1
)
xgb_grid.fit(X_train, y_train)
best_xgb = xgb_grid.best_estimator_
print("Best XGBoost Params:", xgb_grid.best_params_)

In [ ]:
# 5. Evaluate Best Tuned Model
metrics_rf = evaluate(best_rf, X_train, y_train, X_test, y_test, "Tuned Random Forest")
metrics_xgb = evaluate(best_xgb, X_train, y_train, X_test, y_test, "Tuned XGBoost")

df_tuned = pd.DataFrame([metrics_rf, metrics_xgb])
df_tuned

In [ ]:
# Let's print the classification report and plot the confus...
chosen_model = best_xgb
chosen_name = "Tuned XGBoost"

y_pred = chosen_model.predict(X_test)
print(f"Classification Report for {chosen_name}:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f'Confusion Matrix - {chosen_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
# 6. Model Explainability using SHAP
explainer = shap.TreeExplainer(chosen_model)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    shap_val_array = shap_values[1]
else:
    shap_val_array = shap_values

shap.summary_plot(shap_val_array, X_test, plot_type="bar")

In [ ]:
# 6. Model Explainability using SHAP - Part 2
shap.summary_plot(shap_val_array, X_test)

In [ ]:
# 7. Save Final Model
joblib.dump(chosen_model, "../models/churn_model.pkl")
print("Successfully saved best model to models/churn_model.pkl!")